# 模型調整與優化

## 學習目標

完成本 Colab 後，你將能夠：

1. 說明學習率、批次大小、模型複雜度、正則化與優化策略對模型表現的影響。
2. 使用交叉驗證與 Grid Search 進行超參數調校。
3. 比較欠擬合、適度擬合與過擬合模型的差異。
4. 使用正則化與重取樣改善模型泛化能力與類別不平衡問題。
5. 以輕量方式理解模型壓縮與推論加速的概念。

本練習使用 scikit-learn 的內建資料集，不需要下載外部資料，適合在 Google Colab 直接執行。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節需要的 Python 套件，並建立一個可重複實驗的分類資料集。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42

X, y = load_breast_cancer(return_X_y=True)
feature_names = load_breast_cancer().feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)

print("資料筆數:", X.shape[0])
print("特徵數量:", X.shape[1])
print("訓練集大小:", X_train.shape)
print("測試集大小:", X_test.shape)
print("Baseline Accuracy:", round(accuracy_score(y_test, y_pred), 4))


## 核心概念說明

模型調整與優化的目標，不只是讓訓練資料分數變高，而是讓模型在未見過的新資料上也能穩定表現。

### 超參數調校

超參數是訓練前由開發者指定的設定，例如 Logistic Regression 的正則化強度 `C`、決策樹深度、學習率、批次大小等。它們不會由模型自動從資料中學得，但會明顯影響收斂速度、穩定性與泛化能力。

常見做法包含：

- Grid Search：列出候選組合，逐一測試。
- Random Search：隨機抽樣候選組合，適合搜尋空間很大時。
- Cross Validation：用多次切分估計模型是否穩定。

### 正則化

正則化會限制模型參數過度複雜，降低過擬合風險。以 Logistic Regression 為例：

- `C` 較小：正則化較強，模型較保守。
- `C` 較大：正則化較弱，模型較容易貼合訓練資料。

### 重取樣與資料平衡

若某一類資料遠多於另一類，模型可能傾向預測多數類。可透過重取樣、類別權重或評估指標調整來改善。

### 模型壓縮與加速

在實務部署時，模型不只要準，也要能快速推論。可用特徵選擇、降維、剪枝或較小模型，降低資源消耗。


In [ ]:
# ── 示範：使用 Grid Search 調整超參數 ─────────────────
# 這段程式碼示範如何用交叉驗證搜尋 Logistic Regression 的正則化強度與懲罰方式，找出較穩定的設定。

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, solver="liblinear", random_state=RANDOM_STATE))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l1", "l2"]
}

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

results = pd.DataFrame(grid.cv_results_)[[
    "param_model__C",
    "param_model__penalty",
    "mean_train_score",
    "mean_test_score",
    "rank_test_score"
]].sort_values("rank_test_score")

print("最佳參數:", grid.best_params_)
print("交叉驗證最佳分數:", round(grid.best_score_, 4))
print("測試集 Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\n前 5 名參數組合:")
print(results.head())


## 過擬合、欠擬合與正則化

在模型調整時，不能只看訓練分數。常見情境如下：

- 欠擬合：訓練分數與測試分數都低，表示模型能力不足或特徵不足。
- 適度擬合：訓練分數與測試分數都高，且差距不大。
- 過擬合：訓練分數很高，但測試分數明顯較低，表示模型記住訓練資料細節，泛化能力不足。

正則化的核心精神是「不要讓模型為了追求訓練資料的完美表現而變得過度複雜」。在 Logistic Regression 中，`C` 越小，正則化越強；`C` 越大，正則化越弱。


In [ ]:
# ── 示範：比較不同正則化強度 ────────────────────────────
# 這段程式碼比較不同 C 值下的訓練分數與測試分數，觀察正則化強度如何影響泛化能力。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

c_values = [0.001, 0.01, 0.1, 1, 10, 100]
records = []

for c in c_values:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=c, max_iter=3000, random_state=RANDOM_STATE))
    ])
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    records.append({"C": c, "train_accuracy": train_acc, "test_accuracy": test_acc})

result = pd.DataFrame(records)
print(result)

plt.figure(figsize=(8, 4))
plt.plot(result["C"], result["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(result["C"], result["test_accuracy"], marker="o", label="Test Accuracy")
plt.xscale("log")
plt.xlabel("C，數值越小代表正則化越強")
plt.ylabel("Accuracy")
plt.title("正則化強度對模型表現的影響")
plt.legend()
plt.grid(True)
plt.show()


## 實務優化策略

模型調整通常不是單一技巧，而是一組取捨：

1. 若模型不穩定，可以先檢查資料切分、特徵尺度、學習率或正則化。
2. 若模型過擬合，可以增加正則化、降低模型複雜度、加入更多資料或做資料增強。
3. 若資料類別不平衡，可以使用重取樣或類別權重。
4. 若推論太慢，可以減少特徵、改用較小模型或進行模型壓縮。

以下用「類別權重」與「特徵選擇」示範兩個常見的輕量優化方向。


In [ ]:
# ── 實際應用：處理類別不平衡與降低特徵數量 ─────────────────────
# 這段程式碼先建立一個人工不平衡資料集，再比較一般模型與 class_weight='balanced' 的差異，最後示範用 SelectKBest 減少特徵數量以降低模型複雜度。

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, accuracy_score

RANDOM_STATE = 42
X, y = load_breast_cancer(return_X_y=True)

rng = np.random.default_rng(RANDOM_STATE)
majority_idx = np.where(y == 1)[0]
minority_idx = np.where(y == 0)[0]
minority_sample = rng.choice(minority_idx, size=45, replace=False)
imbalanced_idx = np.concatenate([majority_idx, minority_sample])

X_imb = X[imbalanced_idx]
y_imb = y[imbalanced_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=RANDOM_STATE, stratify=y_imb
)

plain_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])

balanced_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE))
])

small_model = Pipeline([
    ("selector", SelectKBest(score_func=f_classif, k=8)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE))
])

for name, model in [
    ("一般模型", plain_model),
    ("加入類別權重", balanced_model),
    ("特徵選擇加類別權重", small_model)
]:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("\n===", name, "===")
    print("使用特徵數:", 8 if name == "特徵選擇加類別權重" else X.shape[1])
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print(classification_report(y_test, y_pred, digits=3))


In [ ]:
# ── 🧪 自我測驗 ──────────────────────────────────
# 請完成下方 TODO 填空，使用 GridSearchCV 找出最佳 C 值，並觀察模型在測試集上的表現。

# TODO: 依提示完成以下程式碼
# 目標：使用 Logistic Regression 調整 C 值，找出最佳超參數。

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])

# TODO 1: 將 C 的候選值設定為 [0.01, 0.1, 1, 10]
param_grid = {
    "model__C": _____
}

# TODO 2: 建立 GridSearchCV，cv 設為 5，scoring 設為 "accuracy"
search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=_____,
    scoring=_____
)

# TODO 3: 使用訓練資料訓練 search
_____.fit(X_train, y_train)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print("最佳 C:", search.best_params_["model__C"])
print("CV Accuracy:", round(search.best_score_, 4))
print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 4))

# Expected: 最佳 C 會是候選值之一，例如 0.1、1 或 10
# Expected: CV Accuracy 與 Test Accuracy 通常會高於 0.95
